# Goal: statistical-summary 

In [13]:
""" 
Goal: statistical-summary
Author: Rudra Prasad Bhuyan
Date: 10-10-2026 14:31 IST
"""

' \nGoal: statistical-summary\nAuthor: Rudra Prasad Bhuyan\nDate: 10-10-2026 14:31 IST\n'

In [14]:
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
from itables import show

In [15]:
path = r"C:\Users\Rudra\Desktop\rural-financial-inclusion-govt-scheme-recommendation\parquet-data\lev-03\data\lev-03_merged.parquet"
pdf = pl.scan_parquet(path)

In [16]:
pdf.collect_schema()

Schema([('Survey_Name', String),
        ('Year', String),
        ('FSU_Serial_No', String),
        ('Sector', String),
        ('State', String),
        ('NSS_Region', String),
        ('District', String),
        ('Stratum', String),
        ('Sub_stratum', String),
        ('Panel', String),
        ('Sub_sample', String),
        ('FOD_Sub_Region', String),
        ('Sample_SU_No', String),
        ('Sample_Sub_Division_No', String),
        ('Second_Stage_Stratum_No', String),
        ('Sample_Household_No', String),
        ('Questionnaire_No', String),
        ('Level', String),
        ('HH_Size_FDQ', Float64),
        ('Engaged_in_Economic_Activity_Las', String),
        ('NCO_2015_Code', String),
        ('NIC_2008_Code', String),
        ('Max_Income_Activity', String),
        ('Self_Employment_Source_Sector', String),
        ('Regular_Wage_Source_Sector', String),
        ('Casual_Labour_Source_Sector', String),
        ('Household_Type', String),
        ('Religion_o

# Useful Variables

In [17]:
lev_03_hh = ['HH_Size_FDQ', 'Household_Type',
            'Religion_of_HH_Head', 'Land_Ownership',
            'Type_of_Land_Owned','Total_Area_Land_Owned_Acres',
            'Energy_Source_Cooking', 'Energy_Source_Lighting',
            'Dwelling_Unit_Exists', 'Type_of_Dwelling_Unit',
            'Rent_Rate_Available_Rural', 'Multiplier']

lev_03_fin = [
    'Self_Employment_Source_Sector','Regular_Wage_Source_Sector',
    'Casual_Labour_Source_Sector', 'Ration_Card_Type']

lev_03_occupation = [
    'Engaged_in_Economic_Activity_Las',	'Max_Income_Activity']

lev_03_scheme = ['Benefitted_From_PMGKY']

lev_03_cols = (
    lev_03_hh
    + lev_03_fin
    + lev_03_occupation
    + lev_03_scheme
)

In [18]:
df = pdf.select(lev_03_cols)

In [19]:
df.head(2).collect()

HH_Size_FDQ,Household_Type,Religion_of_HH_Head,Land_Ownership,Type_of_Land_Owned,Total_Area_Land_Owned_Acres,Energy_Source_Cooking,Energy_Source_Lighting,Dwelling_Unit_Exists,Type_of_Dwelling_Unit,Rent_Rate_Available_Rural,Multiplier,Self_Employment_Source_Sector,Regular_Wage_Source_Sector,Casual_Labour_Source_Sector,Ration_Card_Type,Engaged_in_Economic_Activity_Las,Max_Income_Activity,Benefitted_From_PMGKY
f64,str,str,str,str,str,str,str,str,str,str,f64,str,str,str,str,str,str,str
5.0,"""2""","""1""","""1""","""2""","""1.87""","""02""","""1""","""1""","""1""","""2""",16669.0,"""2""","""""","""""","""3""","""1""","""1""","""2"""
4.0,"""4""","""1""","""1""","""2""","""0.51""","""02""","""1""","""1""","""1""","""2""",24840.0,"""""","""4""","""""","""3""","""1""","""2""","""2"""


In [20]:
df = df.with_columns(
    [pl.col(col).cast(pl.Int32, strict=False) for col in lev_03_cols]
)

In [21]:
unique_counts = df.select(pl.all().n_unique()).collect()
unique_counts

HH_Size_FDQ,Household_Type,Religion_of_HH_Head,Land_Ownership,Type_of_Land_Owned,Total_Area_Land_Owned_Acres,Energy_Source_Cooking,Energy_Source_Lighting,Dwelling_Unit_Exists,Type_of_Dwelling_Unit,Rent_Rate_Available_Rural,Multiplier,Self_Employment_Source_Sector,Regular_Wage_Source_Sector,Casual_Labour_Source_Sector,Ration_Card_Type,Engaged_in_Economic_Activity_Las,Max_Income_Activity,Benefitted_From_PMGKY
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
28,7,9,2,4,1,12,7,2,4,3,23567,3,3,3,7,2,4,2


In [22]:
unique_counts = df.select(pl.all().n_unique()).collect()

categorical_cols = []
numerical_cols = []

for col in df.columns:
    if unique_counts[col][0] <13:
        categorical_cols.append(col)
    else:
        numerical_cols.append(col)


C:\Users\Rudra\AppData\Local\Temp\ipykernel_1688\3703037397.py:6: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  for col in df.columns:


In [23]:
stats = df.select(numerical_cols).describe()
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    display(stats.to_pandas().T)

,0,1,2,3,4,5,6,7,8
statistic,count,null_count,mean,std,min,25%,50%,75%,max
HH_Size_FDQ,1047812.0,0.0,4.218539,2.072699,1.0,3.0,4.0,5.0,31.0
Multiplier,1047812.0,0.0,111344.812944,79655.998345,369.0,56738.0,114044.0,150441.0,2366902.0


In [24]:
for col in categorical_cols:
    print(col)
    counts = df.select(pl.col(col).value_counts(sort=True)).collect()
    
    with pl.Config(tbl_rows=-1, tbl_cols=-1):
        display(counts.unnest(col))

Household_Type


Household_Type,count
i32,u32
1,347200
2,263676
6,107112
9,103780
4,85312
5,80916
3,59816


Religion_of_HH_Head


Religion_of_HH_Head,count
i32,u32
1,812864
2,125872
3,69616
4,16112
6,11260
9,9300
5,2624
0,144
7,20


Land_Ownership


...

Engaged_in_Economic_Activity_Las,count
i32,u32
1,944032
2,103780


Max_Income_Activity


Max_Income_Activity,count
i32,u32
1,438792
2,263860
3,241380
null,103780


Benefitted_From_PMGKY


Benefitted_From_PMGKY,count
i32,u32
1,539012
2,508800
